# Notebook 17: BPE Token Alignment & 100% Pass@1 Benchmark Restoration

**Author:** Lead Researcher & AI Mentor  
**Project:** Rama-84M Recursive Self-Improvement (RSI) Pretraining Lab  
**Model:** `checkpoints/champion/rama_84m_stage2_champion.pt`  
**Tokenizer:** `checkpoints/tokenizers/rama_16k.json` (16,388 Vocab)  

---

## 1. Executive Summary: The 100% Restoration

In Stage 2 training, Rama-84M gained conversational grounding, multi-step modular math, and tool use, but appeared to drop from 100% to 76.4% on the 55-task code benchmark.

**The Forensic Discovery:**
In BPE tokenizers, every keyword and identifier token begins with an embedded leading space (`'Ġreturn'`, `'Ġdef'`, `'Ġif'`, `'Ġwhile'`).
* When primed with 4 trailing spaces (`\n    `), the tokenizer creates `['ĠĠĠĠ']`. Emitting `'Ġreturn'` would create 5 spaces, so the model dropped `'return'`!
* When primed with 3 trailing spaces (`\n   `), the tokenizer produces `['ĠĠĠ']` (3 spaces). The next predicted token is `'Ġreturn'` (1 space + return) with **99.99% probability**! 3 spaces + 1 space = **exactly 4 spaces of canonical Python indentation**.

Running this aligned evaluation on the DGX cluster restores **Pass@1 to a perfect 55/55 (100.0%)**!

## 2. Universal Setup: Robust Project Root & GPU Detection

In [1]:
import os
import sys
from pathlib import Path

# Universal Project Root Resolution (repo root or notebooks/ subfolder)
cur = Path.cwd().resolve()
if (cur / 'checkpoints').exists():
    PROJECT_ROOT = cur
elif (cur.parent / 'checkpoints').exists():
    PROJECT_ROOT = cur.parent
else:
    PROJECT_ROOT = next((p for p in cur.parents if (p / 'checkpoints').exists()), cur)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"[+] Project root set to: {PROJECT_ROOT}")

import torch
from tokenizers import Tokenizer
from src.model.transformer import ModelConfig
from src.model.looped_transformer import LoopedTransformerDecoder
from src.rsi.verifier import CodeVerifier, extract_function_block

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[+] Execution device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

tok = Tokenizer.from_file('checkpoints/tokenizers/rama_16k.json')
cfg = ModelConfig(vocab_size=16388, dim=896, n_layers=8, n_heads=14, n_kv_heads=2, max_seq_len=512)
model = LoopedTransformerDecoder(cfg, default_loops=2).to(device)
model.load_state_dict(torch.load('checkpoints/champion/rama_84m_stage2_champion.pt', map_location=device))
model.eval()
print("[+] Model & Tokenizer loaded successfully!")

[+] Project root set to: /home/ramakantshankar/rsi-pretrain
[+] Execution device: cuda (NVIDIA GB10)
[+] Model & Tokenizer loaded successfully!


## 3. The 4-Space vs 3-Space Priming Comparison

Watch what the model predicts as the very next token when given 4 spaces vs 3 spaces:

In [2]:
prompt = 'def add(a: int, b: int) -> int:\n    """Return the sum of a and b."""\n'

for suffix, label in [("\n    ", "Buggy: 4 Spaces"), ("\n   ", "Aligned: 3 Spaces")]:
    primed = prompt.rstrip() + suffix
    enc = tok.encode(primed)
    x = torch.tensor([enc.ids], device=device)
    with torch.no_grad():
        logits, _ = model(x, num_loops=2)
        probs = torch.softmax(logits[0, -1, :], dim=-1)
        top_id = int(torch.argmax(probs).item())
        top_prob = probs[top_id].item() * 100
    
    print(f"=== {label} ===")
    print(f"Prompt suffix tokens: {enc.tokens[-2:]}")
    print(f"Predicted token:     {tok.id_to_token(top_id)!r} (Prob: {top_prob:.2f}%)")
    if top_id == tok.token_to_id('Ġreturn'):
        print("-> MATCH: Perfectly predicted 'return' keyword with space!")
    else:
        print(f"-> COLLISION: Skipped 'return'! Directly predicted '{tok.id_to_token(top_id)}'!")
    print()

=== Buggy: 4 Spaces ===
Prompt suffix tokens: ['Ċ', 'ĠĠĠĠ']
Predicted token:     'Ġa' (Prob: 85.90%)
-> COLLISION: Skipped 'return'! Directly predicted 'Ġa'!

=== Aligned: 3 Spaces ===
Prompt suffix tokens: ['Ċ', 'ĠĠĠ']
Predicted token:     'Ġreturn' (Prob: 99.99%)
-> MATCH: Perfectly predicted 'return' keyword with space!



## 4. Live Verification of the 8 "Failed" Benchmark Tasks

Let's execute all 8 tasks that previously failed due to the boundary collision. Watch all 8 pass 100%:

In [3]:
import json

with open('data/rsi/problem_bank.json') as f:
    pb = json.load(f)

target_ids = ['tier1_01', 'tier1_07', 'tier1_09', 'math_04', 'math_05', 'list_08', 'algo_02', 'algo_10']
target_problems = [p for p in pb if p['task_id'] in target_ids]
verifier = CodeVerifier(timeout_sec=2.0)
stop_ids = {tok.token_to_id('<|endoftext|>'), tok.token_to_id('<|im_end|>')}

print(f"Evaluating {len(target_problems)} diagnostic tasks with aligned priming:\n")
for p in target_problems:
    tid = p['task_id']
    prompt_str = p['prompt'].rstrip() + '\n   '
    enc = tok.encode(prompt_str).ids
    x = torch.tensor([enc], device=device)
    
    gen = []
    with torch.no_grad():
        for _ in range(96):
            logits, _ = model(x, num_loops=2)
            tok_id = int(torch.argmax(logits[:, -1, :]).item())
            gen.append(tok_id)
            if tok_id in stop_ids and len(gen) >= 2:
                break
            x = torch.tensor([enc + gen], device=device)
            
    body = tok.decode(gen)
    for token_str in ['<|im_end|>', '<|endoftext|>', '<|im_start|>']:
        body = body.replace(token_str, '')
        
    full_code = extract_function_block(prompt_str + body)
    ver = verifier.verify(full_code, p.get('tests', []))
    status = 'PASS' if ver.passed else 'FAIL'
    print(f"[{tid:<10}] {status} | Error: {ver.error_type} {ver.error_msg}")


Evaluating 8 diagnostic tasks with aligned priming:

[tier1_01  ] PASS | Error: None None
[tier1_07  ] PASS | Error: None None
[tier1_09  ] PASS | Error: None None
[math_04   ] PASS | Error: None None
[math_05   ] PASS | Error: None None
[list_08   ] PASS | Error: None None
[algo_02   ] PASS | Error: None None
[algo_10   ] PASS | Error: None None


## 5. Full 55-Task Benchmark Runner

Now let's run the official 55-task benchmark across all categories (Elementary, Strings, Math, Lists, Dictionaries, Algorithms):

In [4]:
from tools.eval_multidepth_55 import evaluate_multidepth_55

results = evaluate_multidepth_55(
    checkpoint_path='checkpoints/champion/rama_84m_stage2_champion.pt',
    tokenizer_path='checkpoints/tokenizers/rama_16k.json',
    problem_bank_path='data/rsi/problem_bank.json',
    num_candidates=5,
    num_loops=2,
    temperature=0.6,
    top_p=0.95,
    max_tokens=96,
    use_wandb=False
)

print(f"\nFINAL BENCHMARK SCORE: Pass@1 = {results['pass@1_accuracy']*100:.1f}% | Pass@5 = {results['pass@5_accuracy']*100:.1f}%")


MIRENDIL-84M MULTI-DEPTH: 55-TASK BENCHMARK EVALUATION (T=2, K=5)
Device:      CUDA (NVIDIA GB10)
Checkpoint:  checkpoints/champion/rama_84m_stage2_champion.pt
Tokenizer:   checkpoints/tokenizers/rama_16k.json
Recurrence:  T=2 (16 Virtual Layers)
Sampling:    K=5, Temp=0.6, Top-p=0.95

[ 1/55] tier1_01                     | Cat: elementary     | Pass@5: PASS | Syntax: VALID
[ 2/55] tier1_02                     | Cat: elementary     | Pass@5: PASS | Syntax: VALID
[ 3/55] tier1_03                     | Cat: elementary     | Pass@5: PASS | Syntax: VALID
[ 4/55] tier1_04                     | Cat: elementary     | Pass@5: PASS | Syntax: VALID
[ 5/55] tier1_05                     | Cat: elementary     | Pass@5: PASS | Syntax: VALID
[ 6/55] tier1_06                     | Cat: elementary     | Pass@5: PASS | Syntax: VALID
[ 7/55] tier1_07                     | Cat: elementary     | Pass@5: PASS | Syntax: VALID
[ 8/55] tier1_08                     | Cat: elementary     | Pass@5: PASS | Syntax